# Feature Enriching Tests 

Adding grid-based and knn-based features to CSV df output from segmentation runs. Testing on both mock data and real CSV output. 

Real file tested (file path): /standard/vol191/siegristlab/Microsam_Segmentation/jobruns/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif/Series001-(3,93,512,512)-dog-1.0-12.0-threshold-0.1/Features_Dpn.csv


### kNN feature enrichment 

function parameters: df, k, radius, cluster quantile

output: df 

features added: 
- nearest_neighbor_label
- nearest_neighbor_distance: distance to single closest object
- knn_mean_distance: average distance to the k nearest neighbors
- knn_density: continuous density score. higher = more locally crowded (closer neighbors), lower = more isolated. calculation: k / mean_distance_to_neighbors. combination of centroid dimensions. 
- neighbors_within_radius
- cluster_flag: 1 if in cluster, 0 if not
- density_threshold_used: dynamically computed from current sample (take top x% densest objects and call them clustered)
- density_quantile_used: more robust than hardcoding a threshold



In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

In [ ]:
def add_neighbor_features(
    df,
    centroid_cols=("centroid-0", "centroid-1", "centroid-2"),
    k=5,
    radius=75,
    cluster_quantile=0.75,
):
    df = df.copy()

    coords = df[list(centroid_cols)].to_numpy(dtype=float)
    labels = df["label"].to_numpy()

    n_neighbors = min(k + 1, len(df))

    nn = NearestNeighbors(n_neighbors=n_neighbors)
    nn.fit(coords)

    distances, indices = nn.kneighbors(coords)

    # remove self neighbor
    neighbor_distances = distances[:, 1:]
    neighbor_indices = indices[:, 1:]

    df["nearest_neighbor_label"] = labels[neighbor_indices[:, 0]]
    df["nearest_neighbor_distance"] = neighbor_distances[:, 0]
    df["knn_mean_distance"] = neighbor_distances.mean(axis=1)
    df["knn_density"] = k / (df["knn_mean_distance"] + 1e-9)

    radius_neighbors = nn.radius_neighbors(coords, radius=radius, return_distance=False)
    df["neighbors_within_radius"] = [len(neigh) - 1 for neigh in radius_neighbors]

    threshold = df["knn_density"].quantile(cluster_quantile)
    df["cluster_flag"] = df["knn_density"] >= threshold
    df["density_threshold_used"] = threshold
    df["density_quantile_used"] = cluster_quantile

    return df

#### Test 1: mock data (using only label + centroids)

Mock centroid data with intentionally structured patterns:
- dense clusters (`cluster_1`, `cluster_2`)
- edge cluster (`edge_cluster`)
- isolated objects

Used to sanity check that kNN and grid-based density features behave as expected before testing on real segmentation outputs

In [ ]:
# create data 
np.random.seed(42)

cluster_1 = np.random.normal(loc=[0, 0, 0], scale=[1, 2, 2], size=(12, 3))
cluster_2 = np.random.normal(loc=[50, 200, 200], scale=[2, 5, 5], size=(10, 3))
edge_cluster = np.random.normal(loc=[400, 50, 50], scale=[2, 4, 4], size=(8, 3))

isolated = np.array([
    [100, 500, 500],
    [150, 600, 600],
    [200, 700, 700],
    [250, 800, 800],
    [300, 900, 900],
    [75, 350, 100],
    [125, 450, 300],
])

coords = np.vstack([cluster_1, cluster_2, edge_cluster, isolated])

df = pd.DataFrame(coords, columns=["centroid-0", "centroid-1", "centroid-2"])
df.insert(0, "label", np.arange(1, len(df) + 1))

df.head()

In [ ]:
# test 
enriched = add_neighbor_features(
    df,
    k=3,
    radius=20,
    cluster_quantile=0.75
)

enriched.sort_values("knn_density", ascending=False).head(15)

#### Test 2: Real segmentation output 

In [ ]:
"""
df = pd.read_csv('/standard/vol191/siegristlab/Microsam_Segmentation/jobruns/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif/Series001-(3,93,512,512)-dog-1.0-12.0-threshold-0.1/Features_Dpn.csv')
df.columns
"""

In [ ]:
# Updated file path 2026-09-28
df = pd.read_csv('/project/vistiqlab/jobruns/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif/Series001-(3,93,512,512)-dog-1.0-12.0-threshold-0.1/Features_Dpn.csv')
df.columns

In [ ]:
df.head()

In [ ]:
enriched = add_neighbor_features(
    df,
    k=5,
    radius=50,
    cluster_quantile=0.75
)
enriched.columns

In [ ]:
print(sum(enriched['cluster_flag']==True))
print(len(enriched))

Note: notably small number of clustered features (71/281)
but 281 is likely too many NBs anyway so good sign to point toward future refiltering/reclassification

### Grid-based Feature Enrichment

function parameters: df, grid_shape 
- df expected columns: label, centroid-0, centroid-1, centroid-2 (x,y,z)

output: df 

features added: 
- grid_z
- grid_y
- grid_x
- grid_cell_count
- grid_density_ratio


assigns grid cells according to `grid_shape`.

`grid_shape=(5,10,10)` means split z into 5 bins, y into 10 bins, x into 10 bins, for a 3D grid 

In [ ]:
# function definition 
def add_grid_density_features(
    df,
    grid_shape=(5, 10, 10),
):

    df = df.copy()

    coords = df[
        ["centroid-0", "centroid-1", "centroid-2"]
    ].to_numpy(dtype=float)

    mins = coords.min(axis=0)
    maxs = coords.max(axis=0)

    ranges = np.where(maxs - mins == 0, 1, maxs - mins)

    normalized = (coords - mins) / ranges

    grid_indices = np.floor(
        normalized * np.array(grid_shape)
    ).astype(int)

    grid_indices = np.minimum(
        grid_indices,
        np.array(grid_shape) - 1
    )

    df["grid_z"] = grid_indices[:, 0]
    df["grid_y"] = grid_indices[:, 1]
    df["grid_x"] = grid_indices[:, 2]

    grid_counts = (
        df.groupby(["grid_z", "grid_y", "grid_x"])
        .size()
        .rename("grid_cell_count")
        .reset_index()
    )

    df = df.merge(
        grid_counts,
        on=["grid_z", "grid_y", "grid_x"],
        how="left"
    )

    total_cells = np.prod(grid_shape)

    avg_objects_per_cell = len(df) / total_cells

    df["grid_density_ratio"] = (
        df["grid_cell_count"] / avg_objects_per_cell
    )

    return df

#### Test 1: mock data 

In [ ]:
# --------------------------------------------------
# Create mock centroid data with obvious structure
# --------------------------------------------------

np.random.seed(42)

# Dense cluster near origin
cluster_1 = np.random.normal(
    loc=[0, 0, 0],
    scale=[1, 2, 2],
    size=(15, 3)
)

# Medium-density cluster
cluster_2 = np.random.normal(
    loc=[50, 200, 200],
    scale=[3, 8, 8],
    size=(12, 3)
)

# Sparse edge cluster
cluster_3 = np.random.normal(
    loc=[400, 50, 50],
    scale=[5, 10, 10],
    size=(8, 3)
)

# Isolated objects
isolated = np.array([
    [100, 500, 500],
    [150, 650, 650],
    [200, 800, 800],
    [250, 950, 950],
    [300, 1100, 1100],
])

# Combine everything
coords = np.vstack([
    cluster_1,
    cluster_2,
    cluster_3,
    isolated
])

# Build dataframe
df = pd.DataFrame(
    coords,
    columns=[
        "centroid-0",
        "centroid-1",
        "centroid-2"
    ]
)

# Add labels
df.insert(0, "label", np.arange(1, len(df) + 1))

print(df.head())
print()
print("Total objects:", len(df))

In [ ]:
# test 
grid_df = add_grid_density_features(
    df,
    grid_shape=(2, 3, 3)
)

grid_df.head(40)

#### Test 2: real segmentation output

__Old path__:
df = pd.read_csv('/standard/vol191/siegristlab/Microsam_Segmentation/jobruns/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif/Series001-(3,93,512,512)-dog-1.0-12.0-threshold-0.1/Features_Dpn.csv')

In [ ]:
df = pd.read_csv('/project/vistiqlab/jobruns/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif/Series001-(3,93,512,512)-dog-1.0-12.0-threshold-0.1/Features_Dpn.csv')

In [ ]:
grid_df_real = add_grid_density_features(
    df,
    grid_shape=(5, 10, 10)
)

grid_df_real.head(40)
#len(grid_df_real)

##### Visualize resulting grid 

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

scatter = ax.scatter(
    grid_df_real["centroid-2"],   # x
    grid_df_real["centroid-1"],   # y
    grid_df_real["centroid-0"],   # z
    c=grid_df_real["grid_cell_count"],
    s=60
)

ax.set_xlabel("x / centroid-2")
ax.set_ylabel("y / centroid-1")
ax.set_zlabel("z / centroid-0")
ax.set_title("Grid Density POC: Objects Colored by Grid Cell Count")

plt.colorbar(scatter, ax=ax, label="Grid cell count")
plt.show()